# Persistence, retries and identity

Read `app/store.py`, `app/auth.py` and the route helpers in `app/main.py`. We use a temporary SQLite database and fictional signing secret. We never import `app.main`, read `.env`, or call a real provider.

Learning goals: distinguish atomicity from idempotency, observe rollback, understand configuration versions and signed claims.


In [1]:
from pathlib import Path
import sys
root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'app' / 'planner.py').exists()), None)
if root is None:
    raise RuntimeError('Open this notebook from the repository or docs/learning directory.')
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
print('Repository imports ready. No provider calls or application server will be started.')


Repository imports ready. No provider calls or application server will be started.


In [2]:
from tempfile import TemporaryDirectory
from app.store import SqliteStore
from app.models import AgentConfig, Interview, Turn, Extraction
from app.goals import GoalAnswer
from app.interview import create_interview, apply_turn
from app.errors import ServiceError

temporary = TemporaryDirectory(prefix='founder-learning-')
store = SqliteStore(str(Path(temporary.name)/'lesson.db'))
original = create_interview('lesson-founder',AgentConfig(),'simulation')
store.update('interviews',original.id,lambda old:original.model_dump())
def submit(turn):
    def mutate(document):
        return apply_turn(Interview.model_validate(document),turn).model_dump()
    return store.update('interviews',original.id,mutate)
turn = Turn(id='stable-request-1',text='Nordlicht',extraction=Extraction(answers=[GoalAnswer(
    goal_key='startup_name',status='answered',value='Nordlicht',quote='Nordlicht')]))
first = submit(turn)
second = submit(turn)
assert first == second
print('One logical turn despite two requests:', len(second['processed_turns']))


One logical turn despite two requests: 1


## Conflicts and rollback

A network retry must preserve both ID and payload. Reusing the ID for different text is a conflict. A mutation that fails must leave the stored document unchanged.


In [3]:
before = store.get('interviews',original.id)
changed = turn.model_copy(update={'text':'Different content'})
try:
    submit(changed)
except ServiceError as exc:
    assert exc.code == 'IDEMPOTENCY_CONFLICT'
    print('Rejected:',exc.code)
assert store.get('interviews',original.id) == before

def broken_mutation(document):
    document['status']='completed'
    raise RuntimeError('Simulated failure before commit')
try:
    store.update('interviews',original.id,broken_mutation)
except RuntimeError:
    pass
assert store.get('interviews',original.id) == before
print('Failed mutation did not change persisted status:', before['status'])


Rejected: IDEMPOTENCY_CONFLICT
Failed mutation did not change persisted status: active


## Optimistic version checks

This small callback demonstrates the same compare-and-increment rule used by the publication endpoint. It is a teaching replica, not a call to the HTTP route. Two editors may read the same version; only the first should publish without reloading.


In [4]:
store.update('config','published',lambda old:{'config':AgentConfig().model_dump(),'versions':[]})
editor_a = store.get('config','published')['config']
editor_b = store.get('config','published')['config']
def publish(draft):
    def change(old):
        if draft['version'] != old['config']['version']:
            raise ServiceError('CONFIG_CONFLICT','Reload before publishing',409)
        return {'config':{**draft,'version':draft['version']+1},'versions':old['versions']+[old['config']]}
    return store.update('config','published',change)
editor_a['name']='New agent'
assert publish(editor_a)['config']['version']==2
try:
    publish(editor_b)
except ServiceError as exc:
    assert exc.code=='CONFIG_CONFLICT'
    print('Stale editor rejected')
assert store.get('interviews',original.id)['config']['name']=='Mira'
print('Existing interview keeps its original configuration')


Stale editor rejected
Existing interview keeps its original configuration


## Signed tokens are not encrypted storage

The token contains a subject, role and expiration. We print decoded fictional claims, not the token. Verification checks signature and expiry; HTTP authorization then checks which operation that role may perform.


In [5]:
from app.config import Settings
from app.auth import Auth
settings = Settings(_env_file=None,environment='development',persistence_backend='sqlite',
                    session_secret='fictional-learning-secret-never-use-in-production')
auth = Auth(settings)
claims = auth.read(auth.token('fictional-founder','founder'))
print({k:claims[k] for k in ['sub','role']})
assert claims['role']=='founder'
try:
    auth.read(auth.token('fictional-founder','founder',lifetime=-1))
except ServiceError as exc:
    assert exc.code=='UNAUTHORIZED'
    print('Expired token rejected')


{'sub': 'fictional-founder', 'role': 'founder'}
Expired token rejected


## Clean up and reflect

SQLite connections are scoped to individual operations, so this temporary directory can be removed. Firestore callbacks may be retried; neither backend transaction can roll back an external provider request. This is why external effects should remain outside mutation callbacks.


In [6]:
temporary.cleanup()
print('Temporary lesson database removed. Existing application data was not modified.')


Temporary lesson database removed. Existing application data was not modified.


## Exercises

1. Change the payload but assign a fresh request ID. Why is that a new logical operation?
2. Describe the difference between an expired session (401), a founder accessing admin settings (403), and an inaccessible interview (404).
3. Why are invitation tokens a different role from logged-in founder tokens?
4. How would you test a successful commit followed by a lost HTTP response?
5. Explain why production needs a stable signing secret and a separate worker credential.
